# Safety Helmet Detection with YOLOv10

Notebook này chạy trọn pipeline huấn luyện và dự đoán mũ bảo hộ trên Google Colab. Các biến cấu hình được gom ở đầu để dễ thay dataset, model hoặc ảnh/video cần dự đoán.

## 1. Cài đặt thư viện

In [ ]:
%pip install -q ultralytics gdown opencv-python

## 2. Cấu hình đường dẫn

Đổi các giá trị trong cell này nếu bạn dùng dataset, model hoặc ảnh khác.

In [ ]:
from pathlib import Path

BASE_DIR = Path('/content')
DATASET_DRIVE_ID = '1l7USE19C7ABIjBjArB4Rb9w3OL6k86xP'
DATASET_ZIP = BASE_DIR / 'Safety_Helmet_Dataset.zip'
DATA_DIR = BASE_DIR / 'safety_helmet_data'
DATA_YAML = DATA_DIR / 'data.yaml'

PRETRAINED_WEIGHTS_URL = 'https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt'
PRETRAINED_WEIGHTS = BASE_DIR / 'yolov10n.pt'

EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16
RUN_NAME = 'helmet-yolov10'

PREDICT_SOURCE = BASE_DIR / 'Rice-Media-construction-worker-helmets-colours-occupation.jpg'
PREDICT_OUTPUT_DIR = BASE_DIR / 'runs' / 'detect' / 'predict'

## 3. Tải weights và dataset

In [ ]:
import urllib.request
import zipfile

import gdown

if not PRETRAINED_WEIGHTS.exists():
    urllib.request.urlretrieve(PRETRAINED_WEIGHTS_URL, PRETRAINED_WEIGHTS)

if not DATASET_ZIP.exists():
    gdown.download(id=DATASET_DRIVE_ID, output=str(DATASET_ZIP), quiet=False)

DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

print(f'Dataset config: {DATA_YAML}')

## 4. Huấn luyện mô hình

In [ ]:
from ultralytics import YOLO

model = YOLO(str(PRETRAINED_WEIGHTS))
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(BASE_DIR / 'runs' / 'detect'),
    name=RUN_NAME,
)

## 5. Đánh giá mô hình

In [ ]:
best_weights = BASE_DIR / 'runs' / 'detect' / RUN_NAME / 'weights' / 'best.pt'
trained_model = YOLO(str(best_weights))
metrics = trained_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, split='test')
metrics

## 6. Chạy dự đoán

In [ ]:
# Thay PREDICT_SOURCE bằng đường dẫn ảnh, video hoặc thư mục bạn muốn dự đoán.
results = trained_model.predict(
    source=str(PREDICT_SOURCE),
    conf=0.25,
    project=str(PREDICT_OUTPUT_DIR.parent),
    name=PREDICT_OUTPUT_DIR.name,
    save=True,
)

print(f'Kết quả đã lưu tại: {PREDICT_OUTPUT_DIR}')

## 7. Hiển thị ảnh kết quả

In [ ]:
from IPython.display import Image, display

if PREDICT_SOURCE.is_file():
    predicted_image = PREDICT_OUTPUT_DIR / PREDICT_SOURCE.name
    display(Image(filename=str(predicted_image)))
else:
    print('Nguồn dự đoán không phải một file ảnh đơn. Hãy mở thư mục output để xem kết quả.')